# Width Metrics Testing Notebook

**Project:** RMIT COSC2408 HN-677 – Pointcloud Analysis for Pedestrian Access with IFP  
**Component:** Width Metrics / Analysis  
**Author:** Sujeeth Gunasekaran

This notebook helps supervisors, clients, and team members test the current width metrics module step-by-step.

It supports:
- Point-based sidewalk width metrics from classified `.laz/.las` files
- Segment-level CSV output review
- JSON summary review
- Basic visualisation of overall width and usable width
- Boundary-based testing when KERB/HFE `.obj` files are available

Current note: the point-based method is ready for experimental testing. The boundary-based method requires boundary OBJ files generated by the boundary extraction workflow.


## 1. Setup

Run this section first. It imports the required libraries and defines the default file paths.

Default point-based input:

`classified/utrecht_mlp_classified.laz`

Expected boundary-based inputs:

`outputs/sidewalk_boundary_kerb.obj`  
`outputs/sidewalk_boundary_hfe.obj`

These lowercase filenames match the current boundary extraction script output.


In [ ]:
from pathlib import Path
import json
import subprocess
import pandas as pd
import matplotlib.pyplot as plt

# Point-based input from the classifier stage
INPUT_FILE = Path("classified/utrecht_mlp_classified.laz")

# Output directory for width metrics
OUTPUT_DIR = Path("outputs/width_metrics")

# Point-based output files
SEGMENT_CSV = OUTPUT_DIR / "sidewalk_segment_metrics.csv"
SUMMARY_JSON = OUTPUT_DIR / "sidewalk_metrics_summary.json"

# Boundary-based output file
BOUNDARY_JSON = OUTPUT_DIR / "boundary_width_summary.json"

# Boundary extraction outputs from the current boundary extraction script
KERB_OBJ = Path("outputs/sidewalk_boundary_kerb.obj")
HFE_OBJ = Path("outputs/sidewalk_boundary_hfe.obj")

print("Point-based input file:", INPUT_FILE)
print("Point-based input exists:", INPUT_FILE.exists())
print("Output directory:", OUTPUT_DIR)
print("Kerb OBJ exists:", KERB_OBJ.exists())
print("HFE OBJ exists:", HFE_OBJ.exists())


## 2. Run Point-Based Width Metrics

This runs the point-based method using the classified `.laz` file.

The method calculates:
- overall sidewalk width
- usable sidewalk width
- obstacle count
- slope estimate
- segment-level metrics

The current implementation uses PCA-based sidewalk direction estimation and percentile-based width ranges to reduce unrealistic raw x/y spread effects.


In [ ]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

command = [
    "python3", "-m", "metrics.width_metrics",
    "--input", str(INPUT_FILE),
    "--output", str(OUTPUT_DIR),
]

result = subprocess.run(command, capture_output=True, text=True)

print(result.stdout)

if result.stderr:
    print("Errors / warnings:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Point-based width metrics command failed.")


## 3. View Point-Based JSON Summary

This displays the summary JSON produced by the point-based workflow.

Important fields:
- `average_overall_width_m`
- `average_usable_width_m`
- `skipped_unrealistic_segments`
- `quality_note`


In [ ]:
if not SUMMARY_JSON.exists():
    raise FileNotFoundError(f"Summary file not found: {SUMMARY_JSON}")

with open(SUMMARY_JSON, "r", encoding="utf-8") as file:
    summary = json.load(file)

summary


## 4. View Segment-Level CSV Results

This loads the detailed segment-level CSV output.


In [ ]:
if not SEGMENT_CSV.exists():
    raise FileNotFoundError(f"Segment CSV not found: {SEGMENT_CSV}")

df = pd.read_csv(SEGMENT_CSV)
df.head(10)


## 5. Segment Statistics

This provides a quick statistical summary of the retained sidewalk segments.


In [ ]:
df.describe()


## 6. Plot Overall Width vs Usable Width

This plot compares detected sidewalk width with estimated usable pedestrian width.

- Overall width = detected sidewalk segment width
- Usable width = estimated walking space after considering obstacle points


In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(df["segment_id"], df["overall_width_m"], label="Overall Width")
plt.plot(df["segment_id"], df["usable_width_m"], label="Usable Width")

plt.xlabel("Segment ID")
plt.ylabel("Width (metres)")
plt.title("Point-Based Sidewalk Width Metrics - Utrecht Test Dataset")
plt.legend()
plt.grid(True)

plt.show()


## 7. Check Largest Retained Widths

This helps identify the largest retained width segments after filtering. These should stay below the reasonable width threshold configured in `metrics/width_metrics.py`.


In [ ]:
df.sort_values("overall_width_m", ascending=False).head(10)


## 8. Run Boundary-Based Width Metrics

This section tests the boundary-based method if the required OBJ files exist.

The expected files are:
- `outputs/sidewalk_boundary_kerb.obj`
- `outputs/sidewalk_boundary_hfe.obj`

These are generated by the boundary extraction script from the classified sidewalk points. If they are missing, this section will skip the boundary test instead of failing the whole notebook.


In [ ]:
print("Kerb OBJ path:", KERB_OBJ)
print("HFE OBJ path:", HFE_OBJ)
print("Kerb OBJ exists:", KERB_OBJ.exists())
print("HFE OBJ exists:", HFE_OBJ.exists())

if KERB_OBJ.exists() and HFE_OBJ.exists():
    command = [
        "python3", "-m", "metrics.width_metrics",
        "--kerb-obj", str(KERB_OBJ),
        "--hfe-obj", str(HFE_OBJ),
        "--output", str(OUTPUT_DIR),
    ]

    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout)

    if result.stderr:
        print("Errors / warnings:")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError("Boundary-based width metrics command failed.")
else:
    print("Boundary OBJ files are missing, so boundary-based testing was skipped.")
    print("Generate or place the files at:")
    print(" -", KERB_OBJ)
    print(" -", HFE_OBJ)


## 9. View Boundary-Based Summary

Run this after the boundary test if the OBJ files were available.


In [ ]:
if BOUNDARY_JSON.exists():
    with open(BOUNDARY_JSON, "r", encoding="utf-8") as file:
        boundary_summary = json.load(file)
    boundary_summary
else:
    print("No boundary summary found yet. Boundary test may have been skipped because OBJ files are missing.")


## 10. Reviewer Notes

Current testing notes:

- The point-based method is ready for experimental review.
- PCA is used to estimate sidewalk direction and avoid raw x/y width distortion.
- Percentile-based ranges reduce the effect of outlier points.
- Segments above the configured reasonable width threshold are excluded from the summary.
- Boundary-based testing requires `sidewalk_boundary_kerb.obj` and `sidewalk_boundary_hfe.obj`.
- The outputs should be treated as analysis/testing results, not final production-grade accessibility measurements yet.
